In [1]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import os
import glob

import spam.DIC
import spam
import spam.plotting

from skimage.io import imread_collection
from skimage import morphology, measure

import pyvista as pv
import gmsh
import meshio

plt.rcParams['font.size'] = 12

In [2]:
tomo = np.zeros((500, 500, 202), dtype=np.bool)
tomo = tomo.astype(np.uint8)
cam_dir = os.path.join(os.environ["HOME"], "work/ssb/output/segmentation/cam")
voids_dir = os.path.join(os.environ["HOME"], "work/ssb/output/segmentation/voids")

In [3]:
for img_id in range(1, 203):
    img_cam = plt.imread(os.path.join(cam_dir, f"{str(img_id).zfill(3)}.tif"))
    # img_voids = plt.imread(os.path.join(voids_dir, f"{str(img_id).zfill(3)}.tif"))
    tomo[np.isclose(img_cam, 2), img_id - 1] = 1
    # tomo[np.isclose(img_voids, 1), img_id - 1] = 0

In [ ]:
binary_labels_spam = spam.label.watershed(tomo)

In [ ]:
radii = spam.label.equivalentRadii(binary_labels_spam)
radii_sieved = np.copy(radii)
radii_sieved[radii_sieved<10] = 0

In [ ]:
# Create the image with each aggregate labelled
spam_sieved = spam.label.convertLabelToFloat(binary_labels_spam, radii_sieved)

# Get new labels for sieved aggregates
spam_sieved_labels = spam.label.watershed(spam_sieved)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(10, 4), dpi=150)

axs[0].imshow(spam_sieved_labels[:, :, 10])
axs[1].imshow(spam_sieved_labels[:, spam_sieved_labels.shape[1]//2, :])
axs[2].imshow(spam_sieved_labels[:, :, spam_sieved_labels.shape[2]//2])

plt.show()

In [ ]:
# Converting the np.array image to a pyvista Uniform Grid
pv_sieved = pv.ImageData()
pv_sieved.dimensions = [500, 500, 202]
# pv_sieved.spacing = [160e-6, 160e-6, 160e-6]
pv_sieved.origin = [0, 0, 0]
pv_sieved.point_data['Label'] = spam_sieved_labels.T.flatten()
agg_flag = np.zeros(spam_sieved_labels.T.shape).flatten()
agg_flag[spam_sieved_labels.T.flatten()!=0] = 1
pv_sieved.point_data['Aggs'] = agg_flag
pv_sieved

In [ ]:
# Saving the aggregates to a vtk file
pv_sieved.save('./output/0_tomo.vtk')

In [ ]:
pv_sieved_e_d = pv_sieved.copy()

n_ero_dil = 2
ks = 5
for i in range(n_ero_dil):
    for j in range(0, 40):
        pv_sieved_e_d = pv_sieved_e_d.image_dilate_erode(dilate_value=0, erode_value=j, kernel_size=(ks, ks, ks))
    
for i in range(n_ero_dil + 1):
    for j in range(0, 40):
        pv_sieved_e_d = pv_sieved_e_d.image_dilate_erode(dilate_value=j, erode_value=0, kernel_size=(ks, ks, ks))

In [ ]:
# Saving the result to a vtk file
pv_sieved_e_d.save('./meshes/1_tomo_e_d.vtk')

In [ ]:
# Creating a common flag for all aggregates to perform the surface meshing
agg_flag_e_d = np.zeros(spam_sieved_labels.T.shape).flatten()
agg_flag_e_d[pv_sieved_e_d.point_data['Label'] != 0] = 1
pv_sieved.point_data['Aggs_e_d'] = agg_flag_e_d

# Applying the marching cubes algorithm
contour = pv_sieved_e_d.contour([1], scalars='Label', method='marching_cubes')

# Saving the result to a vtk file
contour.save('./output/2_contour_raw_mesh.vtk')

In [ ]:
# Creating the raw surface mesh
surf_raw_mesh = contour.extract_geometry()

# Filling the smaller holes using pyvista method
surf_raw_mesh.fill_holes(5, inplace=True)

# Using the clean functionality to remove degenerate surfaces etc
surf_raw_mesh.clean(inplace=True)

In [ ]:
from pymeshfix._meshfix import PyTMesh
mfix = PyTMesh(False)  # False removes extra verbose output
mfix.load_array(surf_raw_mesh.points, surf_raw_mesh.faces.reshape((surf_raw_mesh.n_faces, 4))[:, 1:] )

# Fills all the holes having at at most 'nbe' boundary edges. If
# 'refine' is true, adds inner vertices to reproduce the sampling
# density of the surroundings. Returns number of holes patched.  If
# 'nbe' is 0 (default), all the holes are patched.
mfix.fill_small_boundaries(refine=True)


In [ ]:
# Converting the pymeshfix object to pyvista polydata
vert, faces = mfix.return_arrays()
triangles = np.empty((faces.shape[0], 4), dtype=faces.dtype)
triangles[:, -3:] = faces
triangles[:, 0] = 3

surf_raw_mesh = pv.PolyData(vert, triangles)

In [ ]:
# Splitting the aggregates
aggs_raw = surf_raw_mesh.split_bodies(label=True)
sieved_aggs = []

# Performing the sieving based on the surface area of the aggregates
for agg in aggs_raw:
    if agg.area > 1e-5:
        sieved_aggs.append(agg)
        
sieved_aggs_raw = pv.MultiBlock(sieved_aggs)

In [ ]:
# Performing smoothing of the aggregates and saving each one in an individual stl
for i, sie_agg in enumerate(sieved_aggs):
    print(f'Getting Mesh for Agg: {i+1}')
    sie_agg_raw_surf = sie_agg.extract_geometry()
    sie_agg_smooth_surf = sie_agg_raw_surf.smooth_taubin(n_iter=100, pass_band=0.025 * 2, progress_bar=False)
    pv.save_meshio(f'./output/aggs/agg_{i+1}.stl', sie_agg_smooth_surf)


In [ ]:
# Saving it to a vtk file
sieved_aggs_raw_surf = sieved_aggs_raw.extract_geometry()
surf_smooth_mesh = sieved_aggs_raw_surf.smooth_taubin(n_iter=100, pass_band=0.025 * 2)
surf_smooth_mesh.save('./output/3_surf_smooth_mesh.vtk')

In [ ]:
def group_surfaces_adjacencies(adj):
    '''
    Function to goup the surfaces adjacencies.
    Reference: https://stackoverflow.com/a/4842897
    '''
    l = adj
    out = []
    while len(l)>0:
        first, *rest = l
        first = set(first)

        lf = -1
        while len(first)>lf:
            lf = len(first)

            rest2 = []
            for r in rest:
                if len(first.intersection(set(r)))>0:
                    first |= set(r)
                else:
                    rest2.append(r)     
            rest = rest2

        out.append(list(first))
        l = rest
    return out

In [ ]:
gmsh.initialize()  # Initialize the gmsh API

# Merge each aggregate STL file
for i, agg_path in enumerate(glob.glob('./meshes/aggs/*')):
    print(i)
    gmsh.merge(os.path.join(agg_path))

# Split each surfaces for creating the separated geometry entities
gmsh.model.mesh.classifySurfaces(gmsh.pi, True, True, gmsh.pi)


In [ ]:
# Create a geometry for each one of the discrete entities (aggregates)
gmsh.model.mesh.createGeometry()

In [ ]:
# As the gmsh `classifySurfaces()` function splits the aggregates
# surfaces into multiple parts (most of the time into two parts), retrieve 
# which surfaces are part of which aggregates through adjencies of the curves
surfaces_adjacencies = []
for i, entity in enumerate(gmsh.model.getEntities(1)):
    surfaces_adjacencies.append(gmsh.model.get_adjacencies(entity[0], entity[1])[0])

# Python function to group surfacs that share at least a single upward adjency
surfaces_to_combine = group_surfaces_adjacencies(surfaces_adjacencies)

In [ ]:
# Create a list with the surface loops of each aggregate
volumes = []
agg_surf_loop_list = []
for i, stc in enumerate(surfaces_to_combine):
    agg_surf = gmsh.model.geo.addSurfaceLoop(stc)   # Add the surface loop
    agg_surf_loop_list.append(agg_surf)             # Include in the list
    gmsh.model.geo.addVolume([agg_surf], tag=i)     # Create the volume
    volumes.append(i)
    
# Save the last tag index for the aggregate
agg_last_idx = i

In [ ]:
# Synchronize the built-in CAD representation with the current Gmsh model
gmsh.model.geo.synchronize()

In [ ]:
gmsh.model.addPhysicalGroup(3, volumes, tag=1)
gmsh.model.geo.synchronize()

In [ ]:
gmsh.model.mesh.generate()
# Writing the `.msh` file
gmsh.write("./output/4_final_mesh.msh")
gmsh.finalize()